## Config

In [1]:
import os
import csv
import math
import random
from dataclasses import dataclass
from typing import Dict, List, Tuple

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from torch.amp import GradScaler, autocast

DATA_DIR = 'data'
VOCAB_PATH = os.path.join(DATA_DIR, 'vocab.txt')
META_CSV = os.path.join(DATA_DIR, 'metadata_clean.csv')
TOK_DIR = os.path.join(DATA_DIR, 'tokenized')

# ---------- Training config ----------
SEED = 42
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SEQ_LEN = 50        # input length
STRIDE = 10         # step between windows (reduce redundancy)
BATCH_SIZE = 256
EPOCHS = 8
LR = 1e-3
GRAD_CLIP = 1.0

EMB_DIM = 256
HIDDEN_SIZE = 512
NUM_LAYERS = 2
DROPOUT = 0.2

TRAIN_SPLIT = 0.9   # split by books
MIN_BOOK_TOKENS = 2000  # skip tiny books

STEPS_PER_EPOCH = 3000
CKPT_DIR = 'checkpoints'
SAVE_EVERY_EPOCH = True
SAVE_EVERY_STEPS = 200
# ------------------------------------

use_amp = (DEVICE == "cuda")

print('DEVICE:', DEVICE)
os.makedirs(CKPT_DIR, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

DEVICE: cuda


## Load vocab + metadata

In [2]:
def load_vocab(vocab_path: str) -> Tuple[List[str], Dict[str, int]]:
    if not os.path.exists(vocab_path):
        raise FileNotFoundError(f'vocab.txt not found: {vocab_path}')
    with open(vocab_path, 'r', encoding='utf-8') as f:
        vocab = [line.strip() for line in f if line.strip()]
    tok2id = {t: i for i, t in enumerate(vocab)}
    return vocab, tok2id

def read_metadata(meta_csv: str) -> List[dict]:
    if not os.path.exists(meta_csv):
        raise FileNotFoundError(f'metadata.csv not found: {meta_csv}')
    with open(meta_csv, 'r', encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))

vocab, tok2id = load_vocab(VOCAB_PATH)
rows = read_metadata(META_CSV)

pad_id = tok2id.get('<pad>', 0)
bos_id = tok2id.get('<bos>')
eos_id = tok2id.get('<eos>')

print('vocab size:', len(vocab))
print('special ids:', {'<pad>': pad_id, '<bos>': bos_id, '<eos>': eos_id})
print('metadata rows:', len(rows))

vocab size: 26286
special ids: {'<pad>': 0, '<bos>': 2, '<eos>': 3}
metadata rows: 49


## Dataset (windowed next-token prediction)

In [3]:
def load_ids_file(path: str) -> List[int]:
    with open(path, 'r', encoding='utf-8') as f:
        s = f.read().strip()
    if not s:
        return []
    return list(map(int, s.split()))

@dataclass
class BookData:
    ebook_id: str
    ids_path: str
    token_count: int

def collect_books(rows: List[dict]) -> List[BookData]:
    books = []
    for r in rows:
        ebook_id = r.get('ebook_id')
        if not ebook_id:
            continue
        ids_path = os.path.join(TOK_DIR, f'{ebook_id}.ids.txt')
        if not os.path.exists(ids_path):
            continue
        try:
            token_count = int(r.get('token_count', '0'))
        except Exception:
            token_count = 0
        if token_count < MIN_BOOK_TOKENS:
            continue
        books.append(BookData(str(ebook_id), ids_path, token_count))
    return books

def split_books(books: List[BookData], train_split: float) -> Tuple[List[BookData], List[BookData]]:
    books = books[:]
    random.shuffle(books)
    cut = int(len(books) * train_split)
    return books[:cut], books[cut:]

def load_ids_file(path: str) -> List[int]:
    with open(path, "r", encoding="utf-8") as f:
        s = f.read().strip()
    return list(map(int, s.split())) if s else []

class CachedWindowDataset(Dataset):
    def __init__(self, books, seq_len: int, stride: int):
        self.seq_len = seq_len
        self.stride = stride
        self.book_tokens: List[torch.Tensor] = []
        self.samples: List[Tuple[int, int]] = []

        for b in books:
            ids = load_ids_file(b.ids_path)
            if len(ids) < seq_len + 2:
                continue

            t = torch.tensor(ids, dtype=torch.int32)  # compact in RAM
            bi = len(self.book_tokens)
            self.book_tokens.append(t)

            max_start = t.numel() - (seq_len + 1)
            for st in range(0, max_start + 1, stride):
                self.samples.append((bi, st))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        bi, st = self.samples[idx]
        ids = self.book_tokens[bi]
        chunk = ids[st: st + self.seq_len + 1].to(torch.long)  # convert only this slice
        return chunk[:-1], chunk[1:]

books = collect_books(rows)
if not books:
    raise RuntimeError('No tokenized books found. Check data/tokenized/*.ids.txt and metadata.csv token_count.')

train_books, val_books = split_books(books, TRAIN_SPLIT)
train_ds = CachedWindowDataset(train_books, SEQ_LEN, STRIDE)
val_ds = CachedWindowDataset(val_books, SEQ_LEN, STRIDE)

print(f'Books: total={len(books)} train={len(train_books)} val={len(val_books)}')
print(f'Samples: train={len(train_ds):,} val={len(val_ds):,}')

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,     
    pin_memory=True,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

Books: total=49 train=44 val=5
Samples: train=291,493 val=32,338


## Model

In [4]:
class LSTMLM(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, num_layers: int, dropout: float, pad_id: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        e = self.emb(x)          # (B,T,E)
        out, _ = self.lstm(e)    # (B,T,H)
        out = self.drop(out)
        logits = self.fc(out)    # (B,T,V)
        return logits

## Sampling utilities

In [5]:
@torch.no_grad()
def sample_ids(model: nn.Module, prompt_ids: List[int], max_new_tokens: int = 120,
               temperature: float = 1.0, top_k: int = 50) -> List[int]:
    model.eval()
    x = torch.tensor([prompt_ids], dtype=torch.long, device=DEVICE)
    for _ in range(max_new_tokens):
        logits = model(x)[:, -1, :]  # (1,V)
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.size(-1)))
            probs = torch.softmax(v, dim=-1)
            next_local = torch.multinomial(probs, num_samples=1)
            next_id = ix.gather(-1, next_local)
        else:
            probs = torch.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)

        next_id_int = int(next_id.item())
        x = torch.cat([x, torch.tensor([[next_id_int]], device=DEVICE)], dim=1)

        if eos_id is not None and next_id_int == eos_id:
            break
        if x.size(1) > 512:
            x = x[:, -512:]

    return x[0].tolist()

def decode(ids: List[int]) -> str:
    toks = [vocab[i] if 0 <= i < len(vocab) else '<unk>' for i in ids]
    text = ' '.join(toks)
    text = (text
            .replace(' ,', ',').replace(' .', '.').replace(' !', '!').replace(' ?', '?')
            .replace(' ;', ';').replace(' :', ':')
            .replace(' )', ')').replace('( ', '(')
            .replace(' - ', '-')
           )
    return text

## Train + evaluate

In [6]:
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

def run_eval(model) -> Tuple[float, float]:
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            with autocast("cuda", enabled=use_amp):
                logits = model(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok

    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl

## Tests

In [7]:
def encode_prompt(text: str) -> List[int]:
    toks = text.split()
    ids = [tok2id.get(t, tok2id.get('<unk>', 0)) for t in toks]
    return ids

In [8]:
import itertools

# ---------- Search space ----------
SEARCH_GRID = {
    'lr':          [0.0015, 0.002],
    'hidden_size': [2048],
    'dropout':     [0.5],
}

SEARCH_EMB_DIM    = 256
SEARCH_NUM_LAYERS = 2
SEARCH_EPOCHS     = 8
SEARCH_STEPS_PER_EPOCH = 1500
SEARCH_BATCH_SIZE = 256
SEARCH_SEQ_LEN    = 50
SEARCH_STRIDE     = 10

# Build all configs
keys = list(SEARCH_GRID.keys())
combos = list(itertools.product(*[SEARCH_GRID[k] for k in keys]))
configs = [dict(zip(keys, c)) for c in combos]

print(f"Total configurations: {len(configs)}")
for i, cfg in enumerate(configs):
    print(f"  [{i+1}] lr={cfg['lr']}, hidden={cfg['hidden_size']}, dropout={cfg['dropout']}")

Total configurations: 2
  [1] lr=0.0015, hidden=2048, dropout=0.5
  [2] lr=0.002, hidden=2048, dropout=0.5


In [9]:
def run_search_eval(mdl) -> Tuple[float, float]:
    """Evaluate model on val set, return (avg_loss, perplexity)."""
    mdl.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = criterion(logits.reshape(-1, logits.size(-1)), y.reshape(-1))
            mask = (y != pad_id)
            tok = int(mask.sum().item())
            total_tokens += tok
            total_loss += float(loss.item()) * tok
    avg_loss = total_loss / max(1, total_tokens)
    ppl = math.exp(min(20.0, avg_loss))
    return avg_loss, ppl


search_results = []

for cfg_idx, cfg in enumerate(configs):
    print(f"\n{'='*60}")
    print(f"Config [{cfg_idx+1}/{len(configs)}]  lr={cfg['lr']}  hidden={cfg['hidden_size']}  dropout={cfg['dropout']}")
    print('='*60)

    set_seed(SEED)

    mdl = LSTMLM(
        vocab_size=len(vocab),
        emb_dim=SEARCH_EMB_DIM,
        hidden=cfg['hidden_size'],
        num_layers=SEARCH_NUM_LAYERS,
        dropout=cfg['dropout'],
        pad_id=pad_id,
    ).to(DEVICE)

    crit = nn.CrossEntropyLoss(ignore_index=pad_id)
    opt  = torch.optim.Adam(mdl.parameters(), lr=cfg['lr'])
    scl  = GradScaler("cuda", enabled=use_amp)

    best_val_loss = float('inf')

    for epoch in range(1, SEARCH_EPOCHS + 1):
        mdl.train()
        running = 0.0
        seen_tokens = 0

        pbar = tqdm(enumerate(train_loader, start=1),
                     total=SEARCH_STEPS_PER_EPOCH,
                     desc=f"  E{epoch}/{SEARCH_EPOCHS}", leave=False)

        for step, (x, y) in pbar:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=use_amp):
                logits = mdl(x)
                loss = crit(logits.reshape(-1, logits.size(-1)), y.reshape(-1))

            scl.scale(loss).backward()
            scl.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(mdl.parameters(), GRAD_CLIP)
            scl.step(opt)
            scl.update()

            mask = (y != pad_id)
            tok = int(mask.sum().item())
            running += float(loss.item()) * tok
            seen_tokens += tok

            avg = running / max(1, seen_tokens)
            pbar.set_postfix(loss=f"{avg:.4f}")

            if step >= SEARCH_STEPS_PER_EPOCH:
                break
        pbar.close()

        val_loss, val_ppl = run_search_eval(mdl)
        best_val_loss = min(best_val_loss, val_loss)
        print(f"  epoch {epoch}  train_loss={running/max(1,seen_tokens):.4f}  val_loss={val_loss:.4f}  val_ppl={val_ppl:.2f}")

    search_results.append({
        **cfg,
        'best_val_loss': best_val_loss,
        'best_val_ppl': math.exp(min(20.0, best_val_loss)),
    })

    # free GPU memory before next config
    del mdl, opt, scl, crit
    torch.cuda.empty_cache()

print("\n\nGrid search complete.")


Config [1/2]  lr=0.0015  hidden=2048  dropout=0.5


  E1/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 1  train_loss=4.6163  val_loss=4.1962  val_ppl=66.44


  E2/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 2  train_loss=3.6969  val_loss=4.1542  val_ppl=63.70


...

  epoch 6  train_loss=4.1379  val_loss=4.9609  val_ppl=142.72


  E7/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 7  train_loss=4.0605  val_loss=4.9833  val_ppl=145.96


  E8/8:   0%|          | 0/1500 [00:00<?, ?it/s]

  epoch 8  train_loss=3.9673  val_loss=4.8746  val_ppl=130.93


Grid search complete.


In [10]:
# Sort by best validation loss and display results
search_results.sort(key=lambda r: r['best_val_loss'])

print(f"{'Rank':<5} {'LR':<10} {'Hidden':<8} {'Dropout':<9} {'Val Loss':<10} {'Val PPL':<10}")
print('-' * 55)
for i, r in enumerate(search_results):
    print(f"{i+1:<5} {r['lr']:<10} {r['hidden_size']:<8} {r['dropout']:<9.1f} {r['best_val_loss']:<10.4f} {r['best_val_ppl']:<10.2f}")

print(f"\nBest config:")
best = search_results[0]
print(f"  lr={best['lr']}, hidden_size={best['hidden_size']}, dropout={best['dropout']}")
print(f"  val_loss={best['best_val_loss']:.4f}, val_ppl={best['best_val_ppl']:.2f}")

Rank  LR         Hidden   Dropout   Val Loss   Val PPL   
-------------------------------------------------------
1     0.0015     2048     0.5       4.1542     63.70     
2     0.002      2048     0.5       4.8746     130.93    

Best config:
  lr=0.0015, hidden_size=2048, dropout=0.5
  val_loss=4.1542, val_ppl=63.70
